In [4]:
# %cd "drive/MyDrive/Colab Notebooks/QNLPModelTraining"
import os
import re
import sys
import json
import pickle
import contextlib
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Dict, Optional, Any

import pathlib
from typing import Union
from pathlib import Path
import random



from qiskit_aer import AerSimulator
from pytket.extensions.qiskit.backends.aer import AerBackend
# from qiskit.providers.aer import AerSimulator
# from pytket.extensions.qiskit import AerBackend

from lambeq.backend.grammar import Diagram, Id
from lambeq import (
    QuantumTrainer,
    TketModel, Dataset,
    SPSAOptimizer,
    BinaryCrossEntropyLoss,
    AtomicType, IQPAnsatz,
)



In [26]:
def create_backend_config() -> Dict[str, Any]:
    simulator = AerSimulator(
        method="statevector",
        device="GPU",
        precision="single",
        cuStateVec_enable=True,
    )

    backend = AerBackend(simulation_method="statevector")
    backend._qiskit_backend = simulator

    # print("Available devices:", simulator.available_devices())
    # print("AerSimulator available devices:", AerSimulator().available_devices())
    # print("Backend available devices:", AerBackend()._qiskit_backend.available_devices())

    return {
        "backend": backend,
        "compilation": backend.default_compilation_pass(2),
        "shots": 2,
    }

In [6]:
# from lambeq import (
#     DepCCGParser,
#     # IQPAnsatz,
#     # AtomicType,
#     Rewriter,
#     RemoveCupsRewriter,
#     UnifyCodomainRewriter,
#     SimpleRewriteRule
# )


# ansatz    = IQPAnsatz(
#     {AtomicType.SENTENCE: 1,
#      AtomicType.NOUN:     1, # Galima priskirti 2 qubitus, jei, pvz, treniravimo rezultatai yra prasti
#      }, # AtomicType.PREPOSITIONAL_PHRASE: 0,
#     n_layers=1, n_single_qubit_params=3
# )

# def create_rewriter():
#     # Rule to delete conjunction boxes (“and”, “but”) # just the wire, no box
#     # conj_rule = SimpleRewriteRule(cod=AtomicType.CONJUNCTION, template=Id(AtomicType.CONJUNCTION))

#     rewriter = Rewriter([
#         'determiner',
#         'auxiliary',
#         'connector',
#         'prepositional_phrase',
#     ])
    
#     # rewriter.add_rules(conj_rule)
#     return rewriter

# rewriter = create_rewriter()
# remove_cups = RemoveCupsRewriter()
# unify = UnifyCodomainRewriter(output_type=AtomicType.SENTENCE)

# parser = DepCCGParser(model='elmo', device=0)


In [7]:
# ############### DEBUGGING ##############################

# sentences = [
#     "The company announced a new quantum computing platform.",
#     "The weather was sunny during the event.",
#     "The platform could improve language processing.",
#     "The audience applauded at the end of the presentation.",
# ]
# # sentences = ["Alice likes Bob.","Bob likes Alice.","Alice writes code.","Bob reads books.",]

# labels = np.array([
#     [0, 1],
#     [1, 0],
#     [0, 1],
#     [1, 0],
# ])

# raw_diagrams = parser.sentences2diagrams(
#     sentences,
#     suppress_exceptions=True
# )

# circuits, kept_sentences, kept_labels = [], [], []

# for sentence, diagram, label in zip(sentences, raw_diagrams, labels):
#     if diagram is None:
#         print(f"Skipping failed parse: {sentence}")
#         continue

#     print(sentence)
#     # print("diagram cod:", diagram.cod, " | cod length:", len(diagram.cod))

#     try:
#         diagram = rewriter(diagram)
#         diagram = remove_cups(diagram)
#         diagram = diagram.normal_form()
#         # diagram = diagram.pregroup_normal_form()
#         diagram = unify(diagram)
#         diagram = diagram.normal_form()
        
#         # print("after remove_cups cod:", diagram.cod, " | cod length:", len(diagram.cod))
#         # print()

#         circuit = ansatz(diagram)

#         circuits.append(circuit)
#         kept_sentences.append(sentence)
#         kept_labels.append(label)

#     except Exception as e:
#         print(f"Skipping sentence due to diagram/circuit error: {sentence}")
#         print(type(e).__name__, e)

# kept_labels = np.array(kept_labels)

# for sentence, circuit in zip(kept_sentences, circuits):
#     tk_circuit = circuit.to_tk()
#     # print(sentence)
#     print("qubits:", tk_circuit.n_qubits, "| gates:", tk_circuit.n_gates, "| depth:", tk_circuit.depth())
#     # print()

# # print(f"\nUsable circuits: {len(circuits)}")

# if len(circuits) == 0:
#     raise RuntimeError("No valid circuits were produced.")


# model = TketModel.from_diagrams(
#     circuits,
#     backend_config=backend_config,
# )

# model.initialise_weights()
# outputs = model(circuits)
# # print("\nRaw model outputs:")
# print(outputs)

# ########## DEBUGGING END ###############

In [8]:
from typing import Sequence, Mapping
def get_deep_type(obj):
    if isinstance(obj, list):
        # We look at the unique types inside the list to keep it readable
        inner_types = {get_deep_type(item) for item in obj}
        return f"List[{' | '.join(sorted(inner_types))}]"

    elif isinstance(obj, dict):
        # We summarize the types of all keys and all values
        key_types = {get_deep_type(k) for k in obj.keys()}
        val_types = {get_deep_type(v) for v in obj.values()}
        return f"Dict[{' | '.join(sorted(key_types))}, {' | '.join(sorted(val_types))}]"

    else:
        # Return the class name (e.g., 'Diagram' or 'str')
        return type(obj).__name__

def get_deep_shape(obj, level=0):
    indent = "  " * level
    
    # 1. Atomic types (Strings/Bytes) - Check these first 
    # because they are technically Sequences too!
    if isinstance(obj, (str, bytes)):
        return f"{indent}str"

    # 2. Dictionaries (Mappings)
    elif isinstance(obj, Mapping):
        if not obj:
            return f"{indent}dict(len=0)"

        lines = [f"{indent}dict(len={len(obj)})"]
        for key, value in obj.items():
            # Get child shape and strip only the first line's indentation 
            # so we can prefix it with our key label
            child = get_deep_shape(value, level + 1).lstrip()
            lines.append(f"{indent}  key={repr(key)} -> {child}")
        return "\n".join(lines)

    # 3. Sequences (Lists, Tuples, etc.)
    elif isinstance(obj, Sequence):
        name = type(obj).__name__
        if not obj:
            return f"{indent}{name}(len=0)"

        header = f"{indent}{name}(len={len(obj)})"
        
        # Calculate shapes of all children
        child_shapes = [get_deep_shape(item, level + 1).lstrip() for item in obj]
        unique_shapes = sorted(list(set(child_shapes)))

        if len(unique_shapes) == 1:
            # All items are identical structure
            return f"{header}\n{indent}  [*] -> {unique_shapes[0]}"
        else:
            lines = [header]
            for i, shape in enumerate(child_shapes):
                lines.append(f"{indent}  [{i}] -> {shape}")
            return "\n".join(lines)

    # 4. Base objects (int, float, None, etc.)
    else:
        return f"{indent}{type(obj).__name__}"


In [118]:
EncodedBatch = Dict[str, Any]
EncodedArticle = Dict[str, Any]

def load_encoded_batch(path: Union[str, Path]) -> EncodedBatch:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Pickle file does not exist: {path}")

    if not path.is_file():
        raise ValueError(f"Path is not a file: {path}")

    with open(path, "rb") as f:
        batch = pickle.load(f)

    required_keys = {"encoded_dataset", "errors", "n_qubits"}
    missing = required_keys - set(batch.keys())

    if missing:
        raise KeyError(f"Missing keys in encoded batch {path}: {missing}")

    if not isinstance(batch["encoded_dataset"], list):
        raise TypeError("'encoded_dataset' must be a list of encoded articles.")

    return batch

def load_and_merge_encoded_batches(num_batches: int, start_idx: int = 0) -> EncodedBatch:
    """
    Load a specified number of encoded batch pickle files from 
    Dataset/Encoded/WikiHow/ and merge them into one batch.
    """

    merged_batch: EncodedBatch = {
        "encoded_dataset": [],
        "errors": [],
        "n_qubits": []
    }

    base_path = pathlib.Path("Dataset/Encoded/WikiHow")
    seen_article_ids = set()
    n_sentences = 0
    n_positive_labels = 0

    for i in range(num_batches):
        start = (start_idx + i) * 100
        end = (start_idx + i + 1) * 100
        filename = f"encoded_WikiHow_{start}_{end}.pkl"
        file_path = base_path / filename

        if not file_path.exists():
            print(f"Warning: {file_path} not found. Stopping merge.")
            break

        batch = load_encoded_batch(file_path)

        # Merge dataset and check for duplicates
        for article in batch["encoded_dataset"]:
            article_id = article.get("article_id")
            if article_id in seen_article_ids:
                raise ValueError(
                    f"Duplicate article_id found while merging batches: {article_id}"
                )

            seen_article_ids.add(article_id)
            merged_batch["encoded_dataset"].append(article)
            n_sentences += len(article["circuits"])
            n_positive_labels += article["labels"].count([0,1])

        # Merge metadata lists
        merged_batch["errors"].extend(batch["errors"])
        merged_batch["n_qubits"].extend(batch["n_qubits"])

    print(f"Loaded: {num_batches} batches | {len(seen_article_ids)} articles | {n_sentences} sentences | {n_positive_labels} positive labels")

    return merged_batch

def label_distribution(flat, name: str = "dataset") -> None:
    labels = flat["labels"]

    total = len(labels)
    positives = sum(1 for label in labels if label == [0, 1])
    negatives = sum(1 for label in labels if label == [1, 0])

    if total == 0:
        print(f"{name}: empty")
        return

    # print(f"  total:     {total}")
    print(f"{name}: positive: {positives} ({positives / total:.4f}) | negative: {negatives} ({negatives / total:.4f})")

def split_encoded_batch_aligned(batch: EncodedBatch, train_ratio: float = 0.75, val_ratio: float = 0.10, seed: int = 42, shuffle: bool = True, ) -> Tuple[EncodedBatch, EncodedBatch, EncodedBatch]:
    """
    Split an encoded batch at article level while keeping batch-level metadata aligned.

    Returns:
        train_batch, val_batch, test_batch

    Each returned batch has the same structure:
        {
            "encoded_dataset": List[EncodedArticle],
            "errors": List[List[...]],
            "n_qubits": List[List[int]]
        }
    """

    required_keys = {"encoded_dataset", "n_qubits"}
    missing = required_keys - set(batch.keys())

    if missing:
        raise KeyError(f"Missing keys in batch: {missing}")

    encoded_dataset = batch["encoded_dataset"]
    n_qubits = batch["n_qubits"]

    lengths = {
        "encoded_dataset": len(encoded_dataset),
        "n_qubits": len(n_qubits),
    }

    if len(set(lengths.values())) != 1:
        raise ValueError(
            f"Batch-level lists must have the same length, got: {lengths}"
        )

    if not (0 < train_ratio < 1):
        raise ValueError(f"train_ratio must be between 0 and 1, got {train_ratio}")

    if not (0 <= val_ratio < 1):
        raise ValueError(f"val_ratio must be between 0 and 1, got {val_ratio}")

    if train_ratio + val_ratio >= 1:
        raise ValueError(
            f"train_ratio + val_ratio must be less than 1, got "
            f"{train_ratio + val_ratio}"
        )

    indices = list(range(len(encoded_dataset)))

    if shuffle:
        rng = random.Random(seed)
        rng.shuffle(indices)

    total_count = len(indices)

    train_end = int(total_count * train_ratio)
    val_end = train_end + int(total_count * val_ratio)

    train_indices = indices[:train_end]
    val_indices = indices[train_end:val_end]
    test_indices = indices[val_end:]

    def make_split(split_indices: List[int]) -> EncodedBatch:
        return {
            "encoded_dataset": [encoded_dataset[i] for i in split_indices],
            "n_qubits": [n_qubits[i] for i in split_indices],
        }

    train_batch = make_split(train_indices)
    val_batch = make_split(val_indices)
    test_batch = make_split(test_indices)

    print(
        "Split complete: "
        f"Train={len(train_batch['encoded_dataset'])}, "
        f"Val={len(val_batch['encoded_dataset'])}, "
        f"Test={len(test_batch['encoded_dataset'])}"
    )

    return train_batch, val_batch, test_batch

def flatten_encoded_dataset(encoded_dataset: List[EncodedArticle], *, keep_metadata: bool = True, validate_lengths: bool = True) -> Dict[str, Any]:
    """
    Flatten article-level encoded dataset into sentence/circuit-level lists.

    Input:
        encoded_dataset = [
            {
                "article_id": int,
                "circuits": List[Diagram],
                "labels": List[List[int]],
                "n_qubits": List[int],
                "original_text_sentences": List[str],
            },
            ...
        ]

    Output:
        {
            "circuits": List[Diagram],
            "labels": List[List[int]],
            "sentences": List[str],
            "article_ids": List[int], # not needed
            "sentence_ids": List[int], # not needed
            "n_qubits": List[int], # not needed
        }
    """

    flat = {
        "circuits": [],
        "labels": [],
    }

    if keep_metadata:
        flat.update({
            "article_ids": [],
            "sentence_ids": [],
            "n_qubits": [],
            "sentences": [],
        })

    for article_idx, article in enumerate(encoded_dataset):
        required_keys = {
            "article_id",
            "circuits",
            "labels",
            "n_qubits",
            "original_text_sentences",
        }

        missing = required_keys - set(article.keys())
        if missing:
            raise KeyError(
                f"Article at index {article_idx} is missing keys: {missing}"
            )

        article_id = article["article_id"]
        circuits = article["circuits"]
        labels = article["labels"]
        n_qubits = article["n_qubits"]
        sentences = article["original_text_sentences"]

        if validate_lengths:
            lengths = {
                "circuits": len(circuits),
                "labels": len(labels),
                "n_qubits": len(n_qubits),
                "original_text_sentences": len(sentences),
            }

            unique_lengths = set(lengths.values())

            if len(unique_lengths) != 1:
                raise ValueError(
                    f"Length mismatch in article_id={article_id}: {lengths}"
                )

        for sentence_idx, circuit in enumerate(circuits):
            flat["circuits"].append(circuit)
            flat["labels"].append(labels[sentence_idx])

            if keep_metadata:
                flat["article_ids"].append(article_id)
                flat["sentence_ids"].append(sentence_idx)
                flat["n_qubits"].append(n_qubits[sentence_idx])
                flat["sentences"].append(sentences[sentence_idx])

    return flat



In [ ]:

def binary_accuracy(y_hat, y) -> float:
    """
    Assumes model output is shaped like labels:
        y_hat shape ~= (batch_size, 2)
        y shape     ~= (batch_size, 2)
    """

    y_hat = np.asarray(y_hat)
    y = np.asarray(y)

    pred_classes = np.argmax(y_hat, axis=1)
    true_classes = np.argmax(y, axis=1)

    return np.mean(pred_classes == true_classes)

def validate_ds_data(train_circuits, train_labels, val_circuits, val_labels, ) -> None:
    if len(train_circuits) != len(train_labels):
        raise ValueError(
            f"Train circuits/labels length mismatch: "
            f"{len(train_circuits)} circuits vs {len(train_labels)} labels"
        )

    if len(val_circuits) != len(val_labels):
        raise ValueError(
            f"Validation circuits/labels length mismatch: "
            f"{len(val_circuits)} circuits vs {len(val_labels)} labels"
        )

    if len(train_circuits) == 0:
        raise ValueError("Training set is empty.")

    if len(val_circuits) == 0:
        raise ValueError("Validation set is empty.")

    for i, label in enumerate(train_labels):
        if len(label) != 2:
            raise ValueError(
                f"Expected train label at index {i} to have length 2, got {label}"
            )

    for i, label in enumerate(val_labels):
        if len(label) != 2:
            raise ValueError(
                f"Expected validation label at index {i} to have length 2, got {label}"
            )

def train_quantum_model(
    train_flat: Dict[str, Any],
    val_flat: Dict[str, Any],
    # backend,
    # compilation_pass,
    # *,
    backend_config,
    batch_size: int = 8,
    epochs: int = 5,
    seed: int = 42,
    optimizer_hyperparams: Optional[Dict[str, float]] = None,
    evaluate_on_train: bool = True,
    verbose: str = "text",
) -> Tuple[TketModel, QuantumTrainer]:
    """
    Returns:
        model, trainer

    This function intentionally does NOT include:
        - checkpoint saving
        - staged training
        - resume logic
        - test evaluation
        - custom train/validation schedules
    """

    train_circuits = train_flat["circuits"]
    train_labels = train_flat["labels"]

    val_circuits = val_flat["circuits"]
    val_labels = val_flat["labels"]

    validate_ds_data(train_circuits, train_labels, val_circuits, val_labels)

    # Important:
    # TketModel needs all diagrams it may see, so include both train and validation circuits.
    # Later, when we add test evaluation, we may also include test circuits here.
    all_circuits = train_circuits + val_circuits

    model = TketModel.from_diagrams(
        all_circuits,
        backend_config=backend_config,
    )

    bce = BinaryCrossEntropyLoss()

    if optimizer_hyperparams is None:
        optimizer_hyperparams = {
            "a": 0.05,
            "c": 0.06,
            "A": 0.01 * epochs,
        }

    eval_metrics = {
        "acc": binary_accuracy,
    }

    trainer = QuantumTrainer(
        model=model,
        loss_function=bce,
        epochs=epochs,
        optimizer=SPSAOptimizer,
        optim_hyperparams=optimizer_hyperparams,
        evaluate_functions=eval_metrics,
        evaluate_on_train=evaluate_on_train,
        verbose=verbose,
        seed=seed,
    )

    train_dataset = Dataset(train_circuits, train_labels,
        batch_size=batch_size, shuffle=True,
    )

    val_dataset = Dataset(val_circuits, val_labels,
        batch_size=batch_size, shuffle=False,
    )

    trainer.fit(train_dataset, val_dataset,
        eval_interval=1, log_interval=1,
    )

    return model, trainer


In [ ]:
batch = load_and_merge_encoded_batches(1)
train_data, val_data, test_data = split_encoded_batch_aligned(batch, val_ratio=0.2)

Loaded: 1 batches | 100 articles | 598 sentences | 161 positive labels
Split complete: Train=75, Val=20, Test=5


In [49]:
train_flat = flatten_encoded_dataset(train_data["encoded_dataset"])
val_flat = flatten_encoded_dataset(val_data["encoded_dataset"])
test_flat = flatten_encoded_dataset(test_data["encoded_dataset"])


label_distribution(train_flat, "train")
label_distribution(val_flat, "val")
label_distribution(test_flat, "test")


train: positive: 129 (0.2774) | negative: 336 (0.7226)
val: positive: 26 (0.2364) | negative: 84 (0.7636)
test: positive: 6 (0.2609) | negative: 17 (0.7391)


In [35]:
backend_config = create_backend_config()
backend_config["shots"] = 1024

/home/green/QNLPModelTraining/qnlp_3_10/lib/python3.10/site-packages/pytket/extensions/qiskit/backends/aer.py:129: UserWarning: More than one backend with name 'aer_simulator' is available. Picking one.
  warnings.warn(


In [50]:
model, trainer = train_quantum_model(
    train_flat, val_flat,
    backend_config, batch_size=64,
)

model constructed.


Epoch 1:  train/loss: 1.8997   valid/loss: 1.7935   train/time: 2m21s   valid/time: 7.25s   train/acc: 0.6688   valid/acc: 0.6455
Epoch 2:  train/loss: 2.3409   valid/loss: 1.6778   train/time: 2m20s   valid/time: 7.04s   train/acc: 0.6430   valid/acc: 0.7000
Epoch 3:  train/loss: 1.9390   valid/loss: 2.0033   train/time: 2m23s   valid/time: 7.61s   train/acc: 0.6538   valid/acc: 0.6636
Epoch 4:  train/loss: 1.2208   valid/loss: 2.7237   train/time: 2m26s   valid/time: 6.94s   train/acc: 0.6516   valid/acc: 0.6545
Epoch 5:  train/loss: 1.9812   valid/loss: 1.9412   train/time: 2m20s   valid/time: 6.70s   train/acc: 0.6710   valid/acc: 0.6000

Training completed!
train/time: 11m49s   train/time_per_epoch: 2m22s   train/time_per_step: 17.73s   valid/time: 1m25s   valid/time_per_eval: 8.51s


In [52]:
def evaluate_quantum_predictions(
    model,
    flat_dataset: Dict[str, Any],
    *,
    batch_size: int = 16,
    positive_index: int = 1,
    threshold: Optional[float] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Evaluate a trained lambeq TketModel on a flattened dataset.

    Expected flat_dataset:
        {
            "circuits": List[Diagram],
            "labels": List[List[int]],
            optional:
                "sentences": List[str],
                "article_ids": List[int],
                "sentence_ids": List[int],
                "n_qubits": List[int],
        }

    Label convention:
        [1, 0] = negative / not selected
        [0, 1] = positive / selected

    Prediction convention:
        If threshold is None:
            predicted class = argmax(model_output)

        If threshold is given:
            predicted positive if model_output[:, positive_index] >= threshold

    Returns:
        Dictionary containing predictions, probabilities, and classification metrics.
    """

    circuits = flat_dataset["circuits"]
    labels = flat_dataset["labels"]

    if len(circuits) != len(labels):
        raise ValueError(
            f"Circuits/labels length mismatch: "
            f"{len(circuits)} circuits vs {len(labels)} labels"
        )

    if len(circuits) == 0:
        raise ValueError("Cannot evaluate an empty dataset.")

    y_true_onehot = np.asarray(labels)

    if y_true_onehot.ndim != 2 or y_true_onehot.shape[1] != 2:
        raise ValueError(
            f"Expected labels to have shape (n_samples, 2), "
            f"got {y_true_onehot.shape}"
        )

    y_true = np.argmax(y_true_onehot, axis=1)

    prediction_batches = []

    for start in range(0, len(circuits), batch_size):
        end = start + batch_size
        batch_circuits = circuits[start:end]

        # TketModel.forward(...) calls get_diagram_output(...)
        # and returns an ndarray of model predictions.
        batch_predictions = model.forward(batch_circuits)

        prediction_batches.append(np.asarray(batch_predictions))

    y_prob = np.vstack(prediction_batches)

    if y_prob.ndim != 2 or y_prob.shape[1] != 2:
        raise ValueError(
            f"Expected model predictions to have shape (n_samples, 2), "
            f"got {y_prob.shape}"
        )

    if threshold is None:
        y_pred = np.argmax(y_prob, axis=1)
    else:
        y_pred = (y_prob[:, positive_index] >= threshold).astype(int)

    positive_label = positive_index
    negative_label = 1 - positive_index

    tp = int(np.sum((y_true == positive_label) & (y_pred == positive_label)))
    tn = int(np.sum((y_true == negative_label) & (y_pred == negative_label)))
    fp = int(np.sum((y_true == negative_label) & (y_pred == positive_label)))
    fn = int(np.sum((y_true == positive_label) & (y_pred == negative_label)))

    total = len(y_true)

    accuracy = (tp + tn) / total if total > 0 else 0.0

    precision_pos = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_pos = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_pos = (
        2 * precision_pos * recall_pos / (precision_pos + recall_pos)
        if (precision_pos + recall_pos) > 0
        else 0.0
    )

    precision_neg = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_neg = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_neg = (
        2 * precision_neg * recall_neg / (precision_neg + recall_neg)
        if (precision_neg + recall_neg) > 0
        else 0.0
    )

    positive_count = int(np.sum(y_true == positive_label))
    negative_count = int(np.sum(y_true == negative_label))

    predicted_positive_count = int(np.sum(y_pred == positive_label))
    predicted_negative_count = int(np.sum(y_pred == negative_label))

    majority_baseline_acc = max(positive_count, negative_count) / total

    balanced_accuracy = (recall_pos + recall_neg) / 2

    eps = 1e-12
    clipped_probs = np.clip(y_prob, eps, 1.0 - eps)

    binary_cross_entropy = -np.mean(
        np.sum(y_true_onehot * np.log(clipped_probs), axis=1)
    )

    results = {
        "total": total,

        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "majority_baseline_acc": majority_baseline_acc,
        "binary_cross_entropy": binary_cross_entropy,

        "positive_count": positive_count,
        "negative_count": negative_count,
        "positive_ratio": positive_count / total,
        "negative_ratio": negative_count / total,

        "predicted_positive_count": predicted_positive_count,
        "predicted_negative_count": predicted_negative_count,
        "predicted_positive_ratio": predicted_positive_count / total,
        "predicted_negative_ratio": predicted_negative_count / total,

        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,

        "precision_pos": precision_pos,
        "recall_pos": recall_pos,
        "f1_pos": f1_pos,

        "precision_neg": precision_neg,
        "recall_neg": recall_neg,
        "f1_neg": f1_neg,

        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

    if verbose:
        print("Evaluation results")
        print("------------------")
        print(f"Samples:                  {total}")
        print(f"Accuracy:                 {accuracy:.4f}")
        print(f"Balanced accuracy:        {balanced_accuracy:.4f}")
        print(f"Majority baseline acc:    {majority_baseline_acc:.4f}")
        print(f"Binary cross-entropy:     {binary_cross_entropy:.4f}")
        print()
        print(f"True positives:           {positive_count} ({positive_count / total:.4f})")
        print(f"True negatives:           {negative_count} ({negative_count / total:.4f})")
        print(f"Predicted positives:      {predicted_positive_count} ({predicted_positive_count / total:.4f})")
        print(f"Predicted negatives:      {predicted_negative_count} ({predicted_negative_count / total:.4f})")
        print()
        print("Confusion matrix")
        print(f"TP: {tp} | FP: {fp}")
        print(f"FN: {fn} | TN: {tn}")
        print()
        print("Positive class metrics")
        print(f"Precision:                {precision_pos:.4f}")
        print(f"Recall:                   {recall_pos:.4f}")
        print(f"F1:                       {f1_pos:.4f}")
        print()
        print("Negative class metrics")
        print(f"Precision:                {precision_neg:.4f}")
        print(f"Recall:                   {recall_neg:.4f}")
        print(f"F1:                       {f1_neg:.4f}")

    return results

In [55]:
ds_flat = flatten_encoded_dataset(batch["encoded_dataset"])
label_distribution(ds_flat, "ds_flat")


ds_flat: positive: 161 (0.2692) | negative: 437 (0.7308)


In [ ]:
results = evaluate_quantum_predictions(model, test_flat)

##### debbuging

In [152]:
def validate_circuits(model, ds_flat):
    preds = []
    n_errors = 0
    # try:
    #     preds = model.forward(ds_flat["circuits"], )
    # except Exception as e:
    #     print(f" | {e}")

    for i, circuit in enumerate(tqdm(ds_flat["circuits"])):
        try:
            preds.append(model.forward([circuit]))
        except Exception as e:
            # print(f"{i} | {e}")
            n_errors += 1
    print("invalid circuits:", n_errors)
    return preds


In [ ]:
preds_temp = validate_circuits(model, test_flat)

for pred in preds:
    print(pred)

In [97]:
words = ["sale", "effective", "not", "subject", "photo", "get", "paint", "timed", "again", "studies", "best", "That", "font", "Give", "bound", "keys", "songs", "songs", "quiet", "police", "learn", "unwise", "don"]
ids = []
for i, sent in enumerate(test_flat["sentences"]):
    if any(word in sent for word in words):
        # print(sent)
        # print(test_flat["article_ids"][i])
        ids.append(test_flat["article_ids"][i])

print(ids)

[35, 35, 35, 35, 35, 35, 94, 94, 94, 94, 3, 3, 3, 14, 14, 14, 14, 14, 14, 81, 81, 81, 81]


In [ ]:
article_id = 3
index = test_flat["article_ids"].index(article_id)
# print(len(test_flat["circuits"][index]))
print(len(test_flat["article_ids"]), test_flat["article_ids"])

for i, sent in enumerate(test_flat["sentences"]):
    print(test_flat['article_ids'][i], sent)

In [ ]:
b = load_and_merge_encoded_batches(20, 1)
td, vd, tsd = split_encoded_batch_aligned(b, train_ratio=0.8, val_ratio=0.15)

tf = flatten_encoded_dataset(td["encoded_dataset"])
vf = flatten_encoded_dataset(vd["encoded_dataset"])
tsf = flatten_encoded_dataset(tsd["encoded_dataset"])
bf = flatten_encoded_dataset(b["encoded_dataset"])

label_distribution(tf, "train")
label_distribution(vf, "val")
label_distribution(tsf, "test")

Loaded: 20 batches | 1995 articles | 12264 sentences | 3175 positive labels
Split complete: Train=1596, Val=299, Test=100


In [151]:
p = validate_circuits(model, bf)
print("valid circuits:", len(p))

  0%|          | 0/12264 [00:00<?, ?it/s]

100%|██████████| 12264/12264 [00:57<00:00, 213.88it/s]

12034
valid circuits: 230


In [154]:
save_path = Path("Models/MyModels")
save_path.mkdir(parents=True, exist_ok=True)

with open(save_path / "quantum_model_debugging_notHighProtocol.pkl", "wb") as file:
    pickle.dump(model, file)

In [ ]:
####################################################
####################################################
####################################################
####################################################

In [6]:
def set_stage_configs(shots, epochs, use_stage_configs: bool = False):
    if use_stage_configs:
        stage_configs = []
        for s, a in [(2048, 0.1), (4096, 0.05), (8192, 0.02)]:
            stage_configs.append({
                'epochs': epochs,
                'shots': s,
                'optimizer_hparams': {
                    'a': a,
                    'c': 0.06 if s<=4096 else 0.02,
                    'A': 0.2 * epochs
                }
            })
    else:
        # Single “stage” using default parameters
        stage_configs = [{
            'epochs': epochs,
            'shots': shots,
            'optimizer_hparams': { 'a': 0.1, 'c': 0.06, 'A': 0.2 * epochs }
        }]

    return stage_configs

In [ ]:
def train_quantum_summarizer(
    encoded_train: List[Dict],
    encoded_val:   List[Dict],
    encoded_test:  List[Dict],
    batch_size: int = 5,
    epochs:     int = 10,
    shots:      int = 8192, # 4096
    seed:       int = 42,
    checkpoint_dir:    str = 'saves/model_checkpoints',
    use_stage_configs: bool = True,
    stage_configs:     List[Dict] = None
) -> Tuple[TketModel, QuantumTrainer, Dict[str, float]]:

    os.makedirs(checkpoint_dir, exist_ok=True)

    train_circuits, train_labels = flatten(encoded_train)
    val_circuits,   val_labels   = flatten(encoded_val)
    test_circuits,  test_labels  = flatten(encoded_test)

    if stage_configs is None:
        stage_configs = set_stage_configs(shots, epochs, use_stage_configs)


    final_model   = None
    final_trainer = None
    best_val_acc  = -np.inf
    best_ckpt     = None

    # common metric & datasets
    acc_fn     = lambda y_hat, y: np.mean(np.argmax(y_hat,1) == np.argmax(y,1))
    eval_funcs = {'accuracy': acc_fn}
    val_ds     = Dataset(val_circuits, val_labels, shuffle=False)
    test_ds    = Dataset(test_circuits, test_labels, shuffle=False)


    for idx, cfg in enumerate(stage_configs, start=1):
        t_epochs  = cfg['epochs']
        t_shots   = cfg['shots']
        opt_hp  = cfg['optimizer_hparams']
        batch   = cfg.get('batch_size', batch_size)
        t_seed    = cfg.get('seed', seed + idx)

        print(f"\n=== Stage {idx}: epochs={t_epochs}, shots={t_shots}, batch={batch}, seed={t_seed} ===")

        backend_config = {
            'backend':     backend,
            'compilation': comp_pass,
            'shots':       t_shots,
        }

        ckpt_path = os.path.join(checkpoint_dir, f'model_stage{idx}.lt')

        if final_model is None:
            # First stage: build from scratch
            diagrams = train_circuits + val_circuits
            model = TketModel.from_diagrams(diagrams, backend_config=backend_config)
            model.initialise_weights()
        else:
            # Subsequent stages: resume from last checkpoint
            model = TketModel.from_checkpoint(best_ckpt, backend_config=backend_config)


        trainer = QuantumTrainer(
            model,
            loss_function      = BinaryCrossEntropyLoss(),
            optimizer          = SPSAOptimizer,
            optim_hyperparams  = opt_hp,
            evaluate_functions = eval_funcs,
            evaluate_on_train  = True,
            epochs             = t_epochs,
            seed               = t_seed,
            verbose            = 'text'
        )


        # Fit with early stopping
        train_ds = Dataset(train_circuits, train_labels, batch_size=batch, shuffle=True)
        hist = trainer.fit(
            train_ds,
            val_ds,
            early_stopping_criterion = 'accuracy',
            early_stopping_interval  = 3,
            minimize_criterion       = False
        )

        # Save checkpoint
        model.save(ckpt_path)
        print(f"» Saved checkpoint: {ckpt_path}")

        # 5) Track best val accuracy
        #    hist.val_metrics is a list of dicts of per-epoch val metrics
        #    (you may need to inspect trainer.history if API differs)
        final_val_acc = 0.1
        # final_val_acc = hist.val_metrics[-1]['accuracy']
        if final_val_acc > best_val_acc:
            best_val_acc = final_val_acc
            best_ckpt    = ckpt_path

        # Prepare for next stage
        final_model   = model
        final_trainer = trainer

    test_ds = Dataset(test_circuits, test_labels, shuffle=False)
    test_metrics = final_trainer.evaluate(test_ds)
    print(f"\nFinal test metrics: {test_metrics}")

    return final_model, final_trainer, test_metrics

In [13]:
# Debugging parameters:
model, trainer, metrics = train_quantum_summarizer(
  encoded_train   = ds_test,
  encoded_val     = ds_val,
  encoded_test    = ds_test,

  batch_size = 4,
  epochs     = 1,
  shots      = 32768,
  seed       = 42,
  checkpoint_dir = 'saves/debugging_model_checkpoints',
  use_stage_configs = False
)

# batch_size = 5, epochs = 1, shots = 32, seed = 42,
# train/time: 2m36s   train/time_per_epoch: 2m36s   train/time_per_step: 2.95s   valid/time: 1m16s   valid/time_per_eval: 1m16s

# epochs=1, shots=4, batch=5, seed=42
# train/time: 2m33s   train/time_per_epoch: 2m33s   train/time_per_step: 2.88s   valid/time: 1m15s   valid/time_per_eval: 1m15s

# epochs=1, shots=4, batch=64, seed=42
# train/time: 2m30s   train/time_per_epoch: 2m30s   train/time_per_step: 29.99s   valid/time: 1m14s   valid/time_per_eval: 1m14s

# epochs=1, shots=1, batch=128, seed=42
# train/time: 2m31s   train/time_per_epoch: 2m31s   train/time_per_step: 50.34s   valid/time: 1m14s   valid/time_per_eval: 1m14s




=== Stage 1: epochs=1, shots=32768, batch=4, seed=43 ===
» Saved checkpoint: saves/debugging_model_checkpoints/model_stage1.lt


Epoch 1:  train/loss: 1.4093   valid/loss: 1.1038   train/time: 37.07s   valid/time: 18.83s   train/accuracy: 0.8333   valid/accuracy: 0.8448

Training completed!
train/time: 37.07s   train/time_per_epoch: 37.07s   train/time_per_step: 2.65s   valid/time: 18.83s   valid/time_per_eval: 18.83s


AttributeError: 'QuantumTrainer' object has no attribute 'evaluate'

In [63]:
# Custom 2-stage strategy:
stages = [
  {'epochs': 5,  'shots': 1024,
   'optimizer_hparams': {'a':0.15,'c':0.06,'A':0.2*5},
   'batch_size':  8},
  {'epochs': 15, 'shots': 8192,
   'optimizer_hparams': {'a':0.02,'c':0.02,'A':0.2*15}}
]

model, trainer, metrics = train_quantum_summarizer(
  encoded_train   = ds_test,
  encoded_val     = ds_val,
  encoded_test    = ds_test,

  batch_size = 5,
  seed       = 123,
  # epochs = 10,
  # shots = 8192,
  stage_configs   = stages
)




=== Stage 1: epochs=5, shots=1024, batch=8, seed=124 ===


Epoch 1:  train/loss: 2.4439   valid/loss: 2.8064   train/time: 37.81s   valid/time: 18.21s   train/accuracy: 0.7407   valid/accuracy: 0.7069
Epoch 2:  train/loss: 2.4041   valid/loss: 2.5506   train/time: 38.31s   valid/time: 18.53s   train/accuracy: 0.7963   valid/accuracy: 0.6552
Epoch 3:  train/loss: 0.6235   valid/loss: 1.4374   train/time: 37.50s   valid/time: 18.58s   train/accuracy: 0.7778   valid/accuracy: 0.7586
Epoch 4:  train/loss: 3.9267   valid/loss: 1.3794   train/time: 38.26s   valid/time: 17.26s   train/accuracy: 0.7963   valid/accuracy: 0.8103
Epoch 5:  train/loss: 0.9845   valid/loss: 0.8217   train/time: 36.65s   valid/time: 20.39s   train/accuracy: 0.7593   valid/accuracy: 0.7414

Training completed!
train/time: 3m9s   train/time_per_epoch: 37.71s   train/time_per_step: 5.39s   valid/time: 1m33s   valid/time_per_eval: 18.59s


» Saved checkpoint: saves/model_checkpoints/model_stage1.lt

=== Stage 2: epochs=15, shots=8192, batch=5, seed=125 ===


Epoch 1:   train/loss: 5.5274   valid/loss: 2.1232   train/time: 35.30s   valid/time: 19.44s   train/accuracy: 0.8333   valid/accuracy: 0.7931
Epoch 2:   train/loss: 0.8013   valid/loss: 2.4750   train/time: 35.14s   valid/time: 18.43s   train/accuracy: 0.7407   valid/accuracy: 0.7241
Epoch 3:   train/loss: 5.5922   valid/loss: 2.4570   train/time: 35.00s   valid/time: 17.84s   train/accuracy: 0.8333   valid/accuracy: 0.7069
Epoch 4:   train/loss: 3.1757   valid/loss: 1.3858   train/time: 36.24s   valid/time: 16.85s   train/accuracy: 0.8333   valid/accuracy: 0.8448
Epoch 5:   train/loss: 3.2208   valid/loss: 1.7549   train/time: 36.98s   valid/time: 17.67s   train/accuracy: 0.8704   valid/accuracy: 0.8103
Epoch 6:   train/loss: 3.0971   valid/loss: 1.8732   train/time: 36.44s   valid/time: 17.97s   train/accuracy: 0.7963   valid/accuracy: 0.7069


» Saved checkpoint: saves/model_checkpoints/model_stage2.lt


Epoch 7:   train/loss: 0.6065   valid/loss: 1.4815   train/time: 36.06s   valid/time: 16.87s   train/accuracy: 0.8148   valid/accuracy: 0.7414
Early stopping!
Best model (epoch=4, step=44) saved to
runs/Apr07_19-42-53_DESKTOP-5DCJBRK/best_model.lt

Training completed!
train/time: 4m11s   train/time_per_epoch: 35.88s   train/time_per_step: 3.26s   valid/time: 2m5s   valid/time_per_eval: 17.87s


AttributeError: 'QuantumTrainer' object has no attribute 'evaluate'